In [ ]:
# 1. Import libraries
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import time
import numpy as np

In [ ]:
# ==================================================
# 2. LOAD DATASET
# ==================================================

train = pd.read_csv("dataset/Fashion/fashion-mnist_train.csv")
test = pd.read_csv("dataset/Fashion/fashion-mnist_test.csv")

In [ ]:
# ==================================================
# 3. DATA PREPROCESSING
# ==================================================

# 3.1 Check missing values
print("\nMissing values in training data:", train.isnull().sum().sum())
print("Missing values in testing data:", test.isnull().sum().sum())


# 3.2 Handle missing values
if train.isnull().sum().sum() > 0:
    train = train.fillna(train.median(numeric_only=True))

if test.isnull().sum().sum() > 0:
    test = test.fillna(test.median(numeric_only=True))


# 3.3 Check duplicate rows
print("\nDuplicate rows in training data:", train.duplicated().sum())
print("Duplicate rows in testing data:", test.duplicated().sum())


# Remove duplicates
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)


# 3.4 Display dataset information
print("\nTraining dataset information:")
print(train.info())


# 3.5 Separate features and labels
x_train = train.iloc[:, 1:].values
y_train = train.iloc[:, 0].values

x_test = test.iloc[:, 1:].values
y_test = test.iloc[:, 0].values


# 3.6 Convert data type
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')


# 3.7 Normalize pixel values
x_train = x_train / 255.0
x_test = x_test / 255.0


# 3.8 Display dataset shapes
print("\nTraining data:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing data:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
# ==================================================
# 4. BUILD AND TRAIN 3 MLP MODELS
# ==================================================

histories = []
results = []
times = []
models = []

names = ['Sigmoid', 'Tanh', 'ReLU']

for activation in ['sigmoid', 'tanh', 'relu']:

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(784,)),
        tf.keras.layers.Dense(128, activation=activation),
        tf.keras.layers.Dense(64, activation=activation),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    start = time.time()

    history = model.fit(
        x_train,
        y_train,
        epochs=5,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    times.append(time.time() - start)

    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

    histories.append(history)
    results.append(accuracy)
    models.append(model)

    print(activation, "Accuracy:", accuracy * 100, "%")
    print("Training time:", round(times[-1], 2), "seconds")

In [ ]:
# ==================================================
# 5. TRAINING AND VALIDATION ACCURACY
# ==================================================

plt.figure(figsize=(8,5))

for i, h in enumerate(histories):

    plt.plot(
        h.history['accuracy'],
        label='Model ' + names[i] + ' Training'
    )

    plt.plot(
        h.history['val_accuracy'],
        label='Model ' + names[i] + ' Validation'
    )

plt.title("Accuracy Curve")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ==================================================
# 6. TRAINING AND VALIDATION LOSS
# ==================================================

plt.figure(figsize=(8,5))

for i, h in enumerate(histories):

    plt.plot(
        h.history['loss'],
        label='Model ' + names[i] + ' Training'
    )

    plt.plot(
        h.history['val_loss'],
        label='Model ' + names[i] + ' Validation'
    )

plt.title("Loss Curve")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ==================================================
# 7. TRAINING TIME COMPARISON
# ==================================================

plt.figure(figsize=(8,5))

plt.bar(names, times)

plt.title("Training Time Comparison")
plt.xlabel("Activation Functions")
plt.ylabel("Time (seconds)")
plt.grid(axis='y')
plt.show()

In [ ]:
# ==================================================
# 8. TEST ACCURACY COMPARISON
# ==================================================

print("\nFinal Test Accuracy")

for i, acc in enumerate(results):

    print(
        names[i],
        "Test Accuracy:",
        round(acc * 100, 2),
        "%"
    )

In [ ]:
# ==================================================
# 9. CALCULATE GRADIENT FLOW
# ==================================================

gradient_values = []

for model in models:

    with tf.GradientTape() as tape:

        predictions = model(x_train[:64], training=True)

        loss = tf.keras.losses.sparse_categorical_crossentropy(
            y_train[:64], predictions
        )

        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    gradient_values.append([
    tf.reduce_mean(tf.abs(g)).numpy()
    for g in gradients if g is not None
])

In [ ]:
# ==================================================
# 10. GRADIENT FLOW COMPARISON
# ==================================================

plt.figure(figsize=(8,5))

for i, values in enumerate(gradient_values):

    plt.plot(
        values,
        marker='o',
        label=names[i]
    )

plt.title("Gradient Flow Comparison")
plt.xlabel("Trainable Parameter Tensors")
plt.ylabel("Mean Absolute Gradient")
plt.legend()
plt.grid()
plt.show()


# Find fastest and most accurate model
print("\nModel Comparison")

print("Fastest Model:", names[np.argmin(times)])

print("Most Accurate Model:", names[np.argmax(results)])